[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/raikar8/ML-Lab/blob/main/notebooks/02_triton_flash_attention.ipynb)

# 2 · FlashAttention forward in Triton
Weeks 14–15. `gpu/flash_attention.py` implements the online-softmax tiled attention you wrote in NumPy-style PyTorch in week 14,
now as a real GPU kernel. You'll compare it with a naive implementation that materialises the T×T matrix and with PyTorch's
`scaled_dot_product_attention` (which dispatches to FlashAttention/efficient kernels).

In [ ]:
# Colab: Runtime → Change runtime type → GPU (a T4 is enough except where noted). Kaggle: Settings → Accelerator.
import os, sys
if not os.path.exists("ML-Lab") and not os.path.exists("gpu"):
    !git clone --depth 1 https://github.com/raikar8/ML-Lab.git
if os.path.exists("ML-Lab"):
    %cd ML-Lab
sys.path.insert(0, ".")
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_name() if torch.cuda.is_available() else "NO GPU")
assert torch.cuda.is_available(), "Switch the runtime to a GPU"

In [ ]:
import triton.testing
def bench(fn):  # median milliseconds, with warm-up and L2 flushing handled by Triton
    return triton.testing.do_bench(fn, warmup=25, rep=100)

In [ ]:
from gpu.flash_attention import flash_attention
import torch.nn.functional as F
B, H, D = 4, 8, 64
for causal in (False, True):
    q, k, v = (torch.randn(B, H, 1000, D, device="cuda", dtype=torch.float16) for _ in range(3))
    torch.testing.assert_close(flash_attention(q, k, v, causal), F.scaled_dot_product_attention(q, k, v, is_causal=causal),
                               atol=2e-2, rtol=2e-2)
print("correct")

In [ ]:
def naive(q, k, v, causal):
    s = (q @ k.transpose(-1, -2)) / D ** 0.5
    if causal:
        s = s.masked_fill(torch.ones(s.shape[-2:], device=s.device, dtype=torch.bool).triu(1), float("-inf"))
    return torch.softmax(s, -1) @ v

print(f"{'T':>6} {'naive ms':>9} {'ours ms':>8} {'sdpa ms':>8} {'ours TFLOP/s':>13} {'naive extra MB':>15}")
for T in [512, 1024, 2048, 4096, 8192]:
    q, k, v = (torch.randn(B, H, T, D, device="cuda", dtype=torch.float16) for _ in range(3))
    flops = 4 * B * H * T * T * D / 2  # causal ≈ half
    t_naive = bench(lambda: naive(q, k, v, True)) if T <= 4096 else float("nan")
    t_ours, t_sdpa = bench(lambda: flash_attention(q, k, v, True)), bench(lambda: F.scaled_dot_product_attention(q, k, v, is_causal=True))
    print(f"{T:>6} {t_naive:>9.2f} {t_ours:>8.2f} {t_sdpa:>8.2f} {flops / t_ours / 1e9:>13.1f} {B * H * T * T * 2 / 2**20:>15.0f}")

## Your turn
1. Sweep `BLOCK_M`, `BLOCK_N` ∈ {32, 64, 128} and `num_warps` (pass it in the launch) — find the best config for your GPU.
2. Return the log-sum-exp per row (as in the week-14 exercise) — the backward pass needs it.
3. Implement **FlashDecoding**: for a single query per sequence, split the KV sequence across programs, then combine the partial
   results with the log-sum-exp merge (week 14 `combine_partials`).